![Platform+](images/banner.png)

# Lab 3: Understand Access / RBAC
_Hands-on Snowflake for data practitioners_

## Why can't I see this table? Grant access the right way.

> **Persona track: Platform Admin** — same lab, your angle.

:point_right: **In this lesson, you'll reproduce an access-denied error, understand why it happens, fix it by granting the right privilege, and learn techniques to audit and govern access.**

The scenario: Your coffee roastery has an ORDERS table and a FEEDBACK table. A new analyst role has been set up — but someone forgot to grant access to the feedback data. You'll experience the failure, then fix it — then audit and govern that access.

We'll cover:
1. How access flows in Snowflake: **privilege → role → user**
2. Setting up the lab scenario (tables + partial grants)
3. Reproducing an access-denied error (the #1 new-user question)
4. Granting the missing privilege to fix it
5. Auditing and governing access with SHOW GRANTS
6. Revoking access — the admin's primary governance tool
7. Access review — finding stale grants with ACCOUNT_USAGE
8. Well-Architected governance + Managed Access schemas

:books: **By the end you'll be able to:**
- Explain the full grant chain: warehouse → database → schema → table
- Understand why switching roles can make tables "disappear"
- Grant and revoke SELECT on a table (as the object owner)
- Use SHOW GRANTS to audit access across roles and objects
- Revoke unnecessary privileges to maintain least privilege
- Cross-reference grants with query history to find stale access
- Use Managed Access schemas to centralize control over sensitive data
- Use CoCo as an access governance partner
- Apply the principle of least privilege

**Time:** ~35 min · **Lab interface:** Workspace Notebook with Cortex Code (CoCo) as your tutor.
*Source data lives in the shared, read-only `PLATPLUS_DB`; anything you create lives in your own database (named after your username).*

---
👉 **Connecting to your compute service:**
1. In the top bar, if no service is connected, click to connect and select your service.
2. Set your **role and warehouse** using the picker at the top-right — or just run the setup cell below, which sets them for you in SQL. (SQL cells run on the warehouse.)
3. Once the status shows **Connected** (green), run a cell with **Shift+Enter** (run + advance) or **Cmd/Ctrl+Enter** (run this cell only); the **▶** button works too.

> 📌 The service is a container that runs the notebook kernel and your **Python** cells. **SQL cells (and Snowpark pushdown) run on the query warehouse**, not the container. It stays running until it hits its idle timeout or you suspend it manually.
> 💡 **Best practice: Start a fresh chat for this lab.** Before you begin, open a fresh CoCo chat — click **New chat** (the **+** at the top of the panel). Each lab creates its own objects, so a fresh chat ensures CoCo isn't referencing tables or schemas from a prior lab that no longer exist. Shorter conversations also produce sharper answers and use fewer tokens.

In [ ]:
from snowflake.snowpark.context import get_active_session
session = get_active_session()
user = session.get_current_user().strip('"')
my_db = user + '_DB'
my_wh = user + '_WH'
my_role = user + '_RL'
my_analyst_role = user + '_ANALYST_RL'
print('Context ready. Signed in as ' + user)
print('Your workspace -> db: ' + my_db + ' | wh: ' + my_wh + ' | role: ' + my_role)


## Set your session context

Every Snowflake session operates under a specific role and warehouse. The role determines what you can see and do — the warehouse provides the compute.

> :bulb: **Well-Architected principle:** Never use ACCOUNTADMIN for daily work or object creation. Use the least-privileged role that can accomplish the task.

In [ ]:
USE ROLE {{my_role}};
CREATE WAREHOUSE IF NOT EXISTS {{my_wh}}
  WAREHOUSE_SIZE = XSMALL AUTO_SUSPEND = 60 AUTO_RESUME = TRUE INITIALLY_SUSPENDED = TRUE;
USE WAREHOUSE {{my_wh}};


#### Step 1: Create the tables in your database

In the lab setup below you will copy two tables from the shared `PLATPLUS_DB` into your personal database. We do this so **you own the tables** — which means you can grant privileges on them without needing SECURITYADMIN.

**Why this matters:**
- In Snowflake, the **owner** of an object (the role that created it) can grant privileges on it
- By creating these tables under `{{my_role}}`, you become the owner and can control who else accesses them
- This simulates a real scenario where a team owns their data and manages access to it

| Table | What it contains | Who should have access |
|-------|-----------------|------------------------|
| `ORDERS` | Customer order history | Both {{my_role}} and {{my_analyst_role}} |
| `FEEDBACK` | Customer feedback/reviews | {{my_role}} only (initially) — we'll "forget" to grant {{my_analyst_role}} |


#### Step 2: Grant partial access (deliberately incomplete)

The cell below grants `{{my_analyst_role}}` access to the database, schema, and the ORDERS table — but **intentionally omits** the FEEDBACK table. This sets up the access-denied scenario you'll troubleshoot in Section 3.

**Why we grant in layers (warehouse → database → schema → table):**

Snowflake access is hierarchical. You need grants at **every level** to run a query:

```
Warehouse (USAGE)  →  Database (USAGE)  →  Schema (USAGE)  →  Table (SELECT)
```

Missing any link in the chain = failure. No warehouse = can't execute. No database = can't reference objects. No schema = can't see tables. No SELECT = access denied.

**Best practice:** When onboarding a new role, always grant the full chain. Use a checklist:
1. USAGE on warehouse (provides compute — without this, nothing runs)
2. USAGE on database
3. USAGE on schema
4. Specific privilege on object (SELECT, INSERT, etc.)

> :bulb: In this lab, `{{my_analyst_role}}` already has USAGE on `PLATPLUS_WH` (granted during environment setup by the platform admin). That's why the cell below only grants the database → schema → table chain.

> :wrench: **Real-world note — two paths to compute:**
>
> | Path | When to use | Example |
> |------|-------------|--------|
> | **Use a shared warehouse** | Most common for analysts. A platform admin grants USAGE on an existing warehouse. | `GRANT USAGE ON WAREHOUSE shared_wh TO ROLE analyst_rl;` |
> | **Create your own warehouse** | When you have `CREATE WAREHOUSE` privilege (common for data engineers). You own it and can grant access to others. | `CREATE WAREHOUSE my_team_wh WAREHOUSE_SIZE='XSMALL'; GRANT USAGE ON WAREHOUSE my_team_wh TO ROLE analyst_rl;` |
>
> In this lab we use the shared `PLATPLUS_WH`. In production, data engineers often create dedicated warehouses sized for their workloads (e.g., a small warehouse for ad-hoc queries, a larger one for heavy pipeline loads).

**Use cases for partial access:**
| Scenario | What you grant | What you withhold |
|----------|---------------|-------------------|
| Analyst sees orders but not PII | SELECT on orders | No access to customer_details table |
| Junior dev reads but can't write | SELECT on all tables | No INSERT, UPDATE, DELETE |
| External partner sees summary views | SELECT on views only | No access to base tables |


In [ ]:
CREATE DATABASE IF NOT EXISTS {{my_db}};
CREATE SCHEMA IF NOT EXISTS {{my_db}}.RBAC_LAB;

## 1 · How access works in Snowflake

Access in Snowflake follows a simple chain:

```
PRIVILEGE  →  ROLE  →  USER
(what)        (who)    (person)
```

- **Privileges** are permissions on objects (SELECT on a table, USAGE on a schema)
- **Roles** collect privileges — they define "what this job function can do"
- **Users** are assigned roles — they inherit whatever privileges those roles hold

**The key rule:** In Role-Based Access Control (RBAC), you grant privileges to **roles**. Users then inherit whatever privileges their assigned roles hold.

**Why?** Because people change roles, leave companies, get promoted. If you grant to users, you have to individually manage every person. If you grant to roles, you manage the role once and assign people to it.

> :bulb: **Real-world context:** "Why can't I see this table?" is the #1 support question from new Snowflake users. The answer is almost always: your current role doesn't have a grant on that object. The data didn't disappear — your role just can't see it.

**The system-defined role hierarchy:**
```
ACCOUNTADMIN
  ├── SECURITYADMIN  (manages grants + roles)
  │     └── USERADMIN  (manages users)
  └── SYSADMIN  (manages warehouses + databases)
        └── custom roles (yours!)
              └── PUBLIC  (everyone)
```

Higher roles inherit privileges from lower roles. A custom role should always be granted to SYSADMIN (or another parent) so admins can manage objects created by that role.

## 2 · Set up the lab scenario

As the owner of your database (`{{my_role}}`), you'll create two tables and grant **partial** access to a second role (`{{my_analyst_role}}`). This simulates a real scenario: an analyst is set up with access to orders data, but someone forgot to include the feedback table.

> :bulb: **Where does `{{my_analyst_role}}` come from?** This role was pre-created by your instructor as part of the lab environment setup. It has warehouse usage (so it can run queries) but **no access to any database objects** by default. This is intentional — it represents a new team member's role before anyone has granted them access to specific data. Your job in this lab is to grant (and troubleshoot) that access.

> :point_right: **A note on Sections 1–4:** The RBAC fundamentals (grant chains, access-denied errors, fixing missing privileges) are universal concepts that apply to all personas. Sections 1–4 cover this shared foundation. **Sections 5–8 are where the Platform Admin-specific patterns diverge** — auditing access, revoking privileges, conducting access reviews, and enforcing governance policies that are uniquely part of your role.


In [ ]:
-- Copy two tables into your personal database for the lab
CREATE OR REPLACE TABLE {{my_db}}.RBAC_LAB.ORDERS   AS SELECT * FROM PLATPLUS_DB.ROASTERY_SALES.ORDERS;
CREATE OR REPLACE TABLE {{my_db}}.RBAC_LAB.FEEDBACK AS SELECT * FROM PLATPLUS_DB.ROASTERY_SALES.FEEDBACK;

In [ ]:
-- Warehouse USAGE on PLATPLUS_WH was granted to {{my_analyst_role}} during environment setup.
-- Here we grant the remaining links in the chain: database → schema → table
GRANT USAGE  ON DATABASE {{my_db}}                 TO ROLE {{my_analyst_role}};
GRANT USAGE  ON SCHEMA   {{my_db}}.RBAC_LAB        TO ROLE {{my_analyst_role}};
GRANT SELECT ON TABLE    {{my_db}}.RBAC_LAB.ORDERS TO ROLE {{my_analyst_role}};
-- Note: NO grant on FEEDBACK — that's intentional


> :bulb: **What just happened — decoded:**
>
> | SQL Statement | What it means |
> |---|---|
> | `GRANT USAGE ON WAREHOUSE` | Allows the role to use compute (granted during setup on `PLATPLUS_WH`) |
> | `GRANT USAGE ON DATABASE` | Allows the role to "see" the database exists and enter it |
> | `GRANT USAGE ON SCHEMA` | Allows the role to see the schema and list objects inside |
> | `GRANT SELECT ON TABLE` | Allows the role to query (read) data from that specific table |
>
> **You need ALL FOUR** to query a table. Missing any one link in the chain = failure. The warehouse grant was already in place (from environment setup), so we only needed to add the database → schema → table grants here.

## 3 · Switch roles and hit the wall

Now become `{{my_analyst_role}}` — the limited role. You'll run two queries: one that works and one that fails. The data is the same — the only difference is whether the role has a grant.

**Why this matters (platform admin perspective):**
- **You're the one who gets the access-denied tickets** — When an analyst can't see a table, they come to you. Understanding the failure mode means you can diagnose and resolve it in seconds rather than escalating.
- **You need to verify that access controls are working** — A failed query for an unauthorized role isn't a bug — it's confirmation that your RBAC is doing its job.
- **Context switching is how you test** — As an admin, you regularly switch to other roles to verify what they can and can't see. This is a core audit skill.


In [ ]:
USE ROLE {{my_analyst_role}};
USE WAREHOUSE PLATPLUS_WH;

-- This works: {{my_analyst_role}} has SELECT on ORDERS
SELECT COUNT(*) AS order_count FROM {{my_db}}.RBAC_LAB.ORDERS;


> :white_check_mark: **Why this worked:** In Section 2, you granted `{{my_analyst_role}}` the full access chain — `USAGE` on the database, `USAGE` on the schema, and `SELECT` on ORDERS. All three links are present, so the query succeeds.


> :stop_sign: **The next cell will intentionally fail.** This is expected — it demonstrates what happens when a role is missing a privilege. When you see the error and a **"Fix Error" button appears, do NOT click it.** We want you to understand the error first and fix it yourself in the next section.

In [ ]:
-- This FAILS: {{my_analyst_role}} has NO grant on FEEDBACK
SELECT * FROM {{my_db}}.RBAC_LAB.FEEDBACK LIMIT 5;


> :mag_right: **Read the error message carefully.** You should see something like:
>
> `Object 'FEEDBACK' does not exist or not authorized. Your primary role {{my_analyst_role}} must have at least one privilege granted on TABLE FEEDBACK.`
>
> The table EXISTS. The data is THERE. Your role simply doesn't have a GRANT to read it. This is RBAC working exactly as designed — protecting data from unauthorized access.
>
> **The distinction:** "Does not exist or not authorized" does NOT necessarily mean the table is missing — it may well exist, but your current role can't see it. Snowflake deliberately uses this vague message to avoid leaking information about what objects exist to unauthorized roles.


## 4 · Grant the missing privilege — as the owner

You don't need SECURITYADMIN to fix this. You **own** these tables (you created them with `{{my_role}}`), so you can grant privileges on them.

**Why this matters (platform admin perspective):**
- **You approve and execute access requests** — When teams request access, you evaluate whether it's appropriate, then grant the minimum privilege needed. This section shows the mechanics of that workflow.
- **You enforce the access request process** — In production, no grant happens without a documented request. Understanding the grant syntax means you can process requests quickly and accurately.
- **Least privilege is your responsibility** — You ensure nobody has more access than they need. Granting SELECT (not INSERT/UPDATE/DELETE) is a deliberate choice you make on every request.

---

**Use cases where admins grant access:**
| Scenario | What you grant | Your governance check |
|----------|---------------|----------------------|
| New analyst joins the team | SELECT on approved tables | Verify they need each table, not blanket access |
| Dashboard service account | SELECT on specific views | Confirm it's read-only, no access to base tables |
| Temporary project access | SELECT with a reminder to REVOKE later | Set a calendar reminder for access review |
| Cross-team data sharing | USAGE on schema + SELECT on approved tables | Verify data classification allows sharing |

---

**Best practice — Least privilege:** Grant only the specific privilege needed (SELECT), on the specific object needed (FEEDBACK), to the specific role that needs it ({{my_analyst_role}}). Never grant more than required.

> :bulb: **Well-Architected principle:** In production, access changes should go through a formal request process (e.g., a Jira ticket reviewed by security). Document every grant with: who requested, why, when it should be reviewed/revoked.


In [ ]:
-- Switch back to {{my_role}} (the owner) to grant the missing privilege
USE ROLE {{my_role}};

-- {{my_analyst_role}} already has: WAREHOUSE (USAGE) + DATABASE (USAGE) + SCHEMA (USAGE)
-- The only missing link is SELECT on FEEDBACK
GRANT SELECT ON TABLE {{my_db}}.RBAC_LAB.FEEDBACK TO ROLE {{my_analyst_role}};


In [ ]:
USE ROLE {{my_analyst_role}};
SELECT * FROM {{my_db}}.RBAC_LAB.FEEDBACK LIMIT 5;


> :mag_right: **Same query, same data — the only thing that changed was a GRANT to the role.** That's RBAC in action.
>
> The pattern: 1) User hits access-denied → 2) Identify missing privilege → 3) Grant it to the role → 4) User retries — works.

---
## :robot_face: CoCo Assist

As a platform admin, CoCo isn't just for writing scripts — it's your **access governance partner**. Instead of manually auditing grants across hundreds of objects, you can ask CoCo to build audit queries and identify stale access in seconds. Here are three examples of how you'd use it in the real world.

> :warning: **How to get the most from this section:**
>
> 1. **Read CoCo's response carefully** — don't just glance at it. The explanation is where the learning happens.
> 2. **CoCo is interactive** — if something in the response doesn't make sense, ask a follow-up question. "What do you mean by 'inherited privilege'?" or "Can you show me an example?" The conversation continues until *you* understand.
> 3. **The lab effort is on you** — the prompts below are starting points. Your real learning comes from using the CoCo interface, reading the responses, and asking your own questions based on what you see.
> 4. **There's no single right answer** — CoCo may explain things differently each time. That's fine. Focus on understanding the concept, not memorizing the exact words.

---

**Example 1: Understand RBAC fundamentals**

A colleague asks: *"What's the difference between granting a privilege to a role vs directly to a user?"*

| Term | What it means |
|---|---|
| `GRANT ... TO ROLE` | Assigns a privilege to a role — anyone with that role inherits it |
| `GRANT ... TO USER` | Assigns directly to one person (UBAC) — doesn't scale |
| RBAC (Role-Based) | Standard approach — grant to roles, assign users to roles |
| UBAC (User-Based) | Direct-to-user — useful for private dev, not recommended for production |

🤖 **CoCo Prompt** — *click to copy, then paste into the CoCo panel:*

<pre style="user-select: all; -webkit-user-select: all; background: rgba(41, 181, 232, 0.12); border-left: 4px solid #29B5E8; padding: 12px; border-radius: 0 4px 4px 0; cursor: pointer; white-space: pre-wrap; word-wrap: break-word;">Explain the difference between granting a privilege to a role vs a user in Snowflake. When should I use each approach?</pre>

*Then follow up with:*

<pre style="user-select: all; -webkit-user-select: all; background: rgba(41, 181, 232, 0.12); border-left: 4px solid #29B5E8; padding: 12px; border-radius: 0 4px 4px 0; cursor: pointer; white-space: pre-wrap; word-wrap: break-word;">Give me a real-world scenario where granting directly to a user would be appropriate.</pre>

---

**Example 2: Check your own privileges**

You want to know what access your current role has on the tables you just queried.

🤖 **CoCo Prompt** — *click to copy, then paste into the CoCo panel:*

<pre style="user-select: all; -webkit-user-select: all; background: rgba(41, 181, 232, 0.12); border-left: 4px solid #29B5E8; padding: 12px; border-radius: 0 4px 4px 0; cursor: pointer; white-space: pre-wrap; word-wrap: break-word;">What privileges does {{my_analyst_role}} have on the ORDERS and FEEDBACK tables in {{my_db}}.RBAC_LAB? Show me the grants.</pre>

*Then follow up with:*

<pre style="user-select: all; -webkit-user-select: all; background: rgba(41, 181, 232, 0.12); border-left: 4px solid #29B5E8; padding: 12px; border-radius: 0 4px 4px 0; cursor: pointer; white-space: pre-wrap; word-wrap: break-word;">Why does {{my_analyst_role}} have SELECT but not INSERT? What would happen if I tried to INSERT?</pre>

This is the self-diagnosis skill — instead of guessing why a query failed, you can verify exactly what grants exist (or are missing).

---

**Example 3: Investigate past failures in query history**

Query history logs every query that ran — including failures. To troubleshoot something that *just* failed, use the real-time `INFORMATION_SCHEMA.QUERY_HISTORY` table function — it has no reporting latency, so a failure from moments ago shows up immediately (its `EXECUTION_STATUS` for a failed query is `FAILED_WITH_ERROR`). You can ask CoCo to find recent access-denied errors and explain why they happened.

🤖 **CoCo Prompt** — *click to copy, then paste into the CoCo panel:*

<pre style="user-select: all; -webkit-user-select: all; background: rgba(41, 181, 232, 0.12); border-left: 4px solid #29B5E8; padding: 12px; border-radius: 0 4px 4px 0; cursor: pointer; white-space: pre-wrap; word-wrap: break-word;">Using the real-time INFORMATION_SCHEMA.QUERY_HISTORY table function, check for any queries against the FEEDBACK table in {{my_db}}.RBAC_LAB that failed with an error (EXECUTION_STATUS = 'FAILED_WITH_ERROR') in the last hour. Show me the error messages and explain why they failed.</pre>

*Then follow up with:*

<pre style="user-select: all; -webkit-user-select: all; background: rgba(41, 181, 232, 0.12); border-left: 4px solid #29B5E8; padding: 12px; border-radius: 0 4px 4px 0; cursor: pointer; white-space: pre-wrap; word-wrap: break-word;">What grant would have prevented that failure?</pre>

Query history records the SQL text, user, role, status, and error message for every query executed in the account. Two sources exist: the real-time `INFORMATION_SCHEMA.QUERY_HISTORY` table function used above (best for troubleshooting something that just happened), and the `SNOWFLAKE.ACCOUNT_USAGE.QUERY_HISTORY` view (a 365-day history with up to ~2 hours of latency — better for audits and access reviews). It's invaluable for troubleshooting — you can see exactly what was attempted, by whom, and what went wrong.

> :bulb: **Two things to know about the table function.** It only reaches back **7 days** — query a wider window and it returns an error, so use the `ACCOUNT_USAGE` view for older history. And it returns **only your own queries** unless your role holds `MONITOR`/`OPERATE` on the warehouse (or is `ACCOUNTADMIN`) — a reminder that even query visibility follows RBAC. Here, the failed query was yours, so it shows up regardless of which role is active.

---

> :bulb: **These are just examples.** CoCo can help with any access-related question: "Why can't ROLE_X see this view?", "What's the minimum set of grants for a dashboard to read my table?", "Show me all roles that have access to the SALES schema." The point is that access troubleshooting doesn't have to mean memorizing SHOW GRANTS syntax — you can ask in plain English and follow up until it clicks.


---
## 5 · Your angle — Platform Admin: Audit and govern

The engineer sets up grants as part of deployment. **You audit, review, and govern those grants.** You ensure access is appropriate, not excessive, and compliant.

**Why this matters (platform admin perspective):**
- **You own the access model** — When auditors ask "who can see customer data?", you need to answer definitively. `SHOW GRANTS` is your primary tool.
- **Privilege drift is a real risk** — Over time, grants accumulate. People change teams, projects end, but grants remain. Regular audits catch stale access before it becomes a compliance issue.
- **You distinguish between who HAS access and who SHOULD have access** — Audit queries show the current state. Your governance process decides if it matches policy.

**Use cases for platform admins:**
| Use Case | Command | What you're checking |
|----------|---------|---------------------|
| Who can see this sensitive table? | `SHOW GRANTS ON TABLE x` | All roles with privileges on a specific object |
| What can this role do? | `SHOW GRANTS TO ROLE x` | Full privilege inventory for a role |
| Who has this role? | `SHOW GRANTS OF ROLE x` | Which users/parent roles hold this role |
| Is this access still needed? | Compare grants vs. query history | Identify roles that have access but haven't queried in 90 days |

**The audit commands:**
| Command | What it shows |
|---------|---------------|
| `SHOW GRANTS ON TABLE x` | Who has privileges on a specific table |
| `SHOW GRANTS TO ROLE x` | What privileges a role holds |
| `SHOW GRANTS OF ROLE x` | Which users/parent roles have this role |
| `REVOKE ... FROM ROLE x` | Remove a privilege |

**Best practice:** Run access reviews quarterly. Compare grant lists against team rosters and active projects. If a role hasn't queried a table in 90+ days, that grant is a candidate for revocation.

In [ ]:
USE ROLE {{my_role}};
-- Who has access to FEEDBACK?
SHOW GRANTS ON TABLE {{my_db}}.RBAC_LAB.FEEDBACK;


> :mag_right: **Review.** You should see OWNERSHIP ({{my_role}}) + SELECT ({{my_analyst_role}}).
>
> In production, run this audit regularly to check for **privilege drift** — grants that accumulate beyond what's needed.


In [ ]:
SHOW GRANTS TO ROLE {{my_analyst_role}};


---
### :robot_face: CoCo: Build an access audit query

**Jargon decoder:**
| Term | What it means |
|---|---|
| `SHOW GRANTS ON` | Who has privileges on a specific object |
| `privilege drift` | Grants accumulating beyond what's needed — a security risk |
| `access review` | Periodic check that access matches current needs |

🤖 **CoCo Prompt** — *click to copy, then paste into the CoCo panel:*

<pre style="user-select: all; -webkit-user-select: all; background: rgba(41, 181, 232, 0.12); border-left: 4px solid #29B5E8; padding: 12px; border-radius: 0 4px 4px 0; cursor: pointer; white-space: pre-wrap; word-wrap: break-word;">Write SQL that shows all SELECT grants on all tables in my RBAC_LAB schema, showing which role has access to which table.</pre>

---
## 6 · Extend: Revoke access

Access reviews often reveal stale grants. Revoking is the opposite of granting — and just as important.

**Why this matters (platform admin perspective):**
- **REVOKE is your primary governance tool** — Granting is easy. Revoking takes discipline. A security-conscious organization revokes proactively, not just reactively.
- **Stale access = attack surface** — Every unnecessary grant is a potential data breach vector. If a compromised account has access it doesn't need, the blast radius is larger.
- **Revoke breaks things immediately** — Unlike granting (which is additive), revoking can break downstream queries, dashboards, and pipelines the moment it's executed. Always check dependencies first.

---

**Use cases:**
| Scenario | What you revoke | Impact assessment needed? |
|----------|----------------|-------------------------|
| Employee leaves the company | All grants to their personal role | Low — they're gone |
| Project ends | SELECT grants given for that project | Medium — verify no other project reuses the data |
| Access review finds stale grants | Grants unused for 90+ days | High — check query history to confirm truly unused |
| Security incident | Broad revoke across multiple roles | Critical — coordinate with teams first |

---

**Best practice:** Before revoking in production, always:
1. Check `QUERY_HISTORY` to confirm the grant is truly unused
2. Notify the affected team with a heads-up ("we're revoking X in 7 days unless you confirm you still need it")
3. Revoke during a maintenance window when possible

> :warning: **Be careful with REVOKE in production.** It breaks downstream queries immediately. Always assess impact before executing.

In [ ]:
USE ROLE {{my_role}};
REVOKE SELECT ON TABLE {{my_db}}.RBAC_LAB.FEEDBACK FROM ROLE {{my_analyst_role}};


> :stop_sign: **The next cell will intentionally fail.** This is expected — it proves the REVOKE worked. The "Fix Error" button will appear — **do NOT click it.** The failure IS the expected result.

In [ ]:
USE ROLE {{my_analyst_role}};
SELECT * FROM {{my_db}}.RBAC_LAB.FEEDBACK LIMIT 5;


> :mag_right: **Access denied again.** REVOKE immediately removes the privilege. This confirms revoke works as expected — and this is the state we'll leave the environment in (original baseline restored).

In [ ]:
-- Clean up: ensure baseline is correct (ORDERS accessible, FEEDBACK not)
USE ROLE {{my_role}};

-- Confirm the baseline grants are in place
GRANT USAGE  ON DATABASE {{my_db}}                    TO ROLE {{my_analyst_role}};
GRANT USAGE  ON SCHEMA   {{my_db}}.RBAC_LAB           TO ROLE {{my_analyst_role}};
GRANT SELECT ON TABLE    {{my_db}}.RBAC_LAB.ORDERS    TO ROLE {{my_analyst_role}};
-- FEEDBACK remains revoked — that's the deliberate gap for other persona labs


---
## 7 · Access review: find stale grants with ACCOUNT_USAGE

This is the distinctly admin exercise. Engineers grant access. You **review whether that access is still appropriate** — and revoke what's no longer needed.

The pattern: cross-reference current grants against [`QUERY_HISTORY`](https://docs.snowflake.com/en/sql-reference/account-usage/query_history) to find roles that HAVE access but AREN'T USING it. These are your revocation candidates.

**Why this matters (platform admin perspective):**
- **Compliance requires periodic access reviews** — Auditors ask: "Can you prove everyone who has access actually needs it?" This query provides that evidence.
- **Stale access = unnecessary risk** — A role with SELECT on sensitive data that hasn't queried in 6 months is a dormant threat vector. Revoke it.
- **This is YOUR job, not the engineer's** — Engineers create grants. They don't come back later to clean them up. Access hygiene is an admin responsibility.

**The access review workflow:**
```
1. List all grants on a schema (SHOW GRANTS)
2. Cross-reference with query history (who actually queried these tables?)
3. Identify gaps (roles with grants but no queries)
4. Decision: revoke immediately, or notify the team with a deadline
```

---

**Use cases:**
| Use Case | What you're looking for | Action |
|----------|------------------------|--------|
| Quarterly compliance review | Any role with access that hasn't queried in 90 days | Notify → revoke after grace period |
| Post-project cleanup | Project roles that still exist after project ended | Revoke immediately |
| New hire onboarding audit | Verify new roles only have what they need | Confirm grants match job description |
| Security incident response | Identify all roles with access to compromised data | Assess blast radius |

In [ ]:
-- Access review: which roles have grants on our tables but haven't queried them recently?
USE ROLE {{my_role}};

-- Step 1: See all grants on the FEEDBACK table
SHOW GRANTS ON TABLE {{my_db}}.RBAC_LAB.FEEDBACK;


> :warning: **Latency note:** `SNOWFLAKE.ACCOUNT_USAGE.QUERY_HISTORY` has up to **2 hours of latency**. Queries you ran earlier in this lab session may not appear in the results below. In production this is acceptable — access reviews look at 90-day windows, so a 2-hour delay doesn't matter. But in this lab, you may see fewer rows (or none) than expected. That's normal.

In [ ]:
-- Step 2: Check query history — has {{my_analyst_role}} actually queried FEEDBACK recently?
-- (This simulates what you'd run on any table during an access review)
SELECT
  role_name,
  COUNT(*) AS query_count,
  MAX(start_time) AS last_query_time
FROM SNOWFLAKE.ACCOUNT_USAGE.QUERY_HISTORY
WHERE query_text ILIKE '%RBAC_LAB.FEEDBACK%'
  AND start_time >= DATEADD('day', -90, CURRENT_TIMESTAMP())
  AND role_name = '{{my_analyst_role}}'
GROUP BY role_name
ORDER BY last_query_time DESC;


> :mag_right: **How to read these results:**
>
> **SHOW GRANTS output:** Shows which roles currently have SELECT on FEEDBACK. If {{my_analyst_role}} appears, it has access.
>
> **Query history output:**
> - If {{my_analyst_role}} appears with recent queries → access is actively used. Keep it.
> - If {{my_analyst_role}} appears but the last query was months ago → candidate for revocation.
> - If {{my_analyst_role}} does NOT appear → the role has access but has never used it. Strong revocation candidate.
> - **If no rows returned at all** → this is likely the ACCOUNT_USAGE latency (see the latency note above).
>
> **The admin decision:** Having a grant without using it isn't automatically wrong — some access is precautionary ("just in case"). But you should be able to justify every grant. If you can't explain why a role needs access it hasn't used in 90 days, revoke it.
>
> :bulb: **In production**, you'd run this across all schemas and tables, not just one. CoCo can generate the broader audit queries for you:

🤖 **CoCo Prompt** — *click to copy, then paste into the CoCo panel:*

<pre style="user-select: all; -webkit-user-select: all; background: rgba(41, 181, 232, 0.12); border-left: 4px solid #29B5E8; padding: 12px; border-radius: 0 4px 4px 0; cursor: pointer; white-space: pre-wrap; word-wrap: break-word;">Write a query that finds all roles with SELECT grants on tables in my database that haven't been queried by those roles in the last 90 days.</pre>


---
## 8 · Well-Architected access governance + Managed Access

| Principle | What it means in practice |
|-----------|-------------------------|
| **Least privilege** | Grant only what's needed, nothing more |
| **Role hierarchy** | Separate access roles (permissions) from functional roles (people) |
| **Regular access reviews** | Audit grants quarterly; revoke what's stale (Section 7) |
| **Managed access schemas** | For sensitive data, centralize grant control (demo below) |
| **No ACCOUNTADMIN for objects** | Never create tables with ACCOUNTADMIN |
| **Future grants** | Automate access for new objects (the engineer's job — you govern it) |

---

### Managed Access schemas — centralized control

In a normal schema, the **table owner** can grant access to their own tables. This means any engineer who creates a table can share it with anyone — which is fine for analytics data, but dangerous for sensitive data (PII, financials, health records).

A **Managed Access schema** changes the rules: only the **schema owner** (you, the admin) can grant access to objects inside it — even if someone else created the table.

**Why this matters (platform admin perspective):**
- **Centralized control over sensitive data** — Engineers can create tables in the schema, but they can't share them without your approval
- **Prevents accidental data exposure** — A junior engineer can't accidentally grant SELECT on a PII table to a reporting role
- **Compliance requirement** — Many regulations require that access to sensitive data is controlled by a designated data steward (you)

**Normal schema vs Managed Access schema:**
| | Normal Schema | Managed Access Schema |
|---|---|---|
| Who can grant on objects? | Object owner (whoever created it) | Schema owner only (the admin) |
| Who can create objects? | Anyone with the relevant CREATE privilege (e.g. CREATE TABLE, CREATE VIEW) | Same — no change |
| Use case | Analytics, shared data | Sensitive data, PII, regulated data |
| Admin control | Distributed (each owner manages their own) | Centralized (admin manages all) |

Let's create one and see the difference:

In [ ]:
-- Step 1: Admin creates a Managed Access schema
USE ROLE {{my_role}};
CREATE OR REPLACE SCHEMA {{my_db}}.MANAGED_LAB WITH MANAGED ACCESS;

-- Step 2: Admin grants {{my_analyst_role}} permission to CREATE tables in it
GRANT USAGE ON SCHEMA {{my_db}}.MANAGED_LAB TO ROLE {{my_analyst_role}};
GRANT CREATE TABLE ON SCHEMA {{my_db}}.MANAGED_LAB TO ROLE {{my_analyst_role}};


> :bulb: **Now switch to `{{my_analyst_role}}`.** You just granted it `CREATE TABLE` on the managed schema, so it can create — and own — a table there. The next cell runs as that role.

In [ ]:
-- Step 3: {{my_analyst_role}} creates a table (this works — they have CREATE TABLE)
USE ROLE {{my_analyst_role}};
USE WAREHOUSE PLATPLUS_WH;
CREATE OR REPLACE TABLE {{my_db}}.MANAGED_LAB.MY_REPORT (id INT, data STRING);
INSERT INTO {{my_db}}.MANAGED_LAB.MY_REPORT VALUES (1, 'sensitive data');


> :mag_right: **{{my_analyst_role}} just created a table in a Managed Access schema.** They OWN the table. But can they share it with another role?
>
> In a **normal schema**: yes — table owners can grant SELECT on their own tables to any role.
> In a **Managed Access schema**: NO — only the schema owner (the admin) can grant, regardless of who created the table.
>
> The next cell proves this. {{my_analyst_role}} will try to grant SELECT on their own table to the PUBLIC role (a role that everyone has) — and it will **fail**.

> :stop_sign: **The next cell will intentionally fail.** This is the "aha" moment — the table owner can't share their own data. Only the admin can. Do NOT click Fix Error.


In [ ]:
-- {{my_analyst_role}} tries to grant SELECT on their OWN table to another role — this FAILS in managed access
-- ({{my_analyst_role}} owns the table but cannot share it — only the schema owner can)
USE ROLE {{my_analyst_role}};
GRANT SELECT ON TABLE {{my_db}}.MANAGED_LAB.MY_REPORT TO ROLE PUBLIC;


> :white_check_mark: **Proof:** {{my_analyst_role}} **owns** the table but **cannot grant** access to it. In a managed access schema, only the schema owner controls sharing.
>
> **This is the admin's power:** Engineers can build whatever they need in the schema, but they can't accidentally (or intentionally) expose sensitive data to unauthorized roles. All sharing goes through you.
>
> **Compare to Section 4:** In `RBAC_LAB` (a normal schema), {{my_role}} could grant SELECT on tables it owned — because normal schemas allow table owners to manage their own grants. Managed Access schemas remove that ability for everyone except the schema owner.


### Cleanup — managed access in action, even during teardown

**Why cleanup needs two roles:** The table `MY_REPORT` inside the managed access schema is owned by `{{my_analyst_role}}` — only that role can drop it. Once the table is gone, the schema owner (`{{my_role}}`) can drop the now-empty schema.

This is managed access enforcing boundaries even during teardown: no single role can unilaterally clean up everything. In production, this is exactly why admins keep an inventory of who owns what inside managed schemas.

In [ ]:
-- Clean up: drop the managed access schema (lab artifact)
-- Step 1: Switch to the table owner — only they can DROP their own table
USE ROLE {{my_analyst_role}};
DROP TABLE IF EXISTS {{my_db}}.MANAGED_LAB.MY_REPORT;
-- Step 2: Switch to the schema owner — only they can DROP the schema
USE ROLE {{my_role}};
DROP SCHEMA IF EXISTS {{my_db}}.MANAGED_LAB;


---
## Verify — Original state restored

The cleanup restored the environment to the **true original state** from Section 2: {{my_analyst_role}} has ORDERS but NOT FEEDBACK.

1. ORDERS query succeeds (baseline grant intact)
2. FEEDBACK query fails (deliberate gap restored by the REVOKE in Section 6)

> :stop_sign: **The second verification cell will intentionally fail** — this proves the cleanup worked and the environment is ready for other persona labs.


In [ ]:
-- Verify Part 1: ORDERS is accessible (should succeed)
USE ROLE {{my_analyst_role}};
SELECT COUNT(*) AS orders_count FROM {{my_db}}.RBAC_LAB.ORDERS;


In [ ]:
-- Verify Part 2: FEEDBACK is NOT accessible (should fail — proves original state restored)
SELECT COUNT(*) AS feedback_count FROM {{my_db}}.RBAC_LAB.FEEDBACK;


---
### Check your understanding 🧠
A few questions on this lab's key themes. Run each quiz cell below, choose an answer, and review the feedback.

In [ ]:
from resources.knowledgecheck import platform_admin_quiz as quiz
quiz.display_question("q1")

In [ ]:
from resources.knowledgecheck import platform_admin_quiz as quiz
quiz.display_question("q2")

In [ ]:
from resources.knowledgecheck import platform_admin_quiz as quiz
quiz.display_question("q3")

In [ ]:
from resources.knowledgecheck import platform_admin_quiz as quiz
quiz.display_question("q4")

---
### :snowflake: Key Takeaways

- Privileges go to **roles**, roles go to users — that's RBAC
- The full grant chain: **Warehouse → Database → Schema → Table** — missing any link = access denied
- "Does not exist or not authorized" does NOT necessarily mean the object is missing — it may exist, but your role can't see it
- An object **owner** can grant and revoke privileges on what they own — **unless** it's a Managed Access schema
- **`SHOW GRANTS ON TABLE`** shows who has access to a specific object — your primary audit tool
- **`SHOW GRANTS TO ROLE`** shows everything a role can do — your access review tool
- **REVOKE removes access immediately** — always check dependencies before revoking in production
- **Cross-reference grants with query history** to find stale access (roles with grants but no queries)
- **Managed Access schemas** centralize grant control — only the schema owner can grant, even on tables others created
- Regular access reviews catch privilege drift before it becomes a compliance issue
- The engineer sets up grants as part of deployment — **you audit, review, and govern** those grants
- **CoCo is your governance partner** — ask it to generate audit queries, compare access, or identify stale grants

**The Other Personas:**
- The **Data Analyst** notebook covers introspecting access and secondary roles
- The **Data Engineer** notebook covers future grants, schema-wide automation, and deployment workflows

> :bulb: Each persona lab is independent — you can run them in any order. The cleanup cell restores the environment to baseline between labs.

### Notebook completion. 🏁
You're done — your notebook is saved automatically.

> 💡 **Moving on to another lab?** Shut down this notebook's **kernel** — click **Connected** at the top → **Shut down kernel**. All your lab notebooks share one compute service, so clearing a finished kernel frees up memory for the next lab. Your service stays running, ready in seconds.

> 💡 **Finished for the day?** **Suspend your service** — click **Connected** → hover your service name → **Suspend**. This stops compute; it resumes automatically next time you run a cell.